# Demo: SEC EDGAR Data Pipeline

**Module:** `src.common.ingestion`  
**Purpose:** Download 10-K filings from SEC EDGAR, extract key sections, and save as structured Markdown.  
**Shared by:** All 4 RAG systems (ensures fair comparison).

### Why Markdown?

Markdown was chosen as the **unified data format** for all 4 systems to minimize format-induced bias:
- **RAG systems** (1 & 2): `##` headers enable semantic chunking (split at section boundaries)
- **Long-Context systems** (3 & 4): Minimal token overhead vs. JSON syntax
- **Neutral**: Doesn't advantage either architecture class

---

## 1. Setup & Configuration

The pipeline uses `configs/base.yaml` for shared settings and `.env` for secrets (API keys, SEC identity).  
All config is loaded through `src.common.config.load_config()`.

In [7]:
import logging
from src.common import load_config, setup_logging

setup_logging(logging.INFO)

# Show current config (secrets are injected from .env)
config = load_config()
print("LLM Model:", config['llm']['model'])
print("Embedding Model:", config['embedding']['model'])
print("Target Companies:", [c['ticker'] for c in config['companies']])

LLM Model: gemini-2.0-flash
Embedding Model: text-embedding-004
Target Companies: ['AAPL', 'MSFT', 'AMZN']


## 2. Download a Single 10-K Filing

The `download_filing()` function:
1. Connects to SEC EDGAR via `edgartools`
2. Finds the latest 10-K filing for a given ticker
3. Extracts the full text and individual sections (Items 1-15)
4. Saves:
   - `data/raw/{TICKER}/10K_{date}.txt` — Raw text from EDGAR
   - `data/processed/{TICKER}/10K_{date}.md` — Structured Markdown with `##` section headers
   - `data/processed/{TICKER}/10K_{date}.meta.json` — Machine-readable metadata sidecar

In [8]:
from src.common import download_filing

# Download Apple's latest 10-K
filing = download_filing("AAPL")

print(f"Company: {filing.metadata.company_name}")
print(f"Filing Date: {filing.metadata.filing_date}")
print(f"CIK: {filing.metadata.cik}")
print(f"Full Text Length: {len(filing.full_text):,} characters")
print(f"Sections Extracted: {len(filing.sections)}")

2026-02-28 13:08:33 [INFO] edgar.core: Identity of the Edgar REST client set to [daniel.kojok@studmail.w-hs.de]
2026-02-28 13:08:33 [INFO] src.common.ingestion: EDGAR identity set to: daniel.kojok@studmail.w-hs.de
2026-02-28 13:08:33 [INFO] src.common.ingestion: Fetching 10-K filing for AAPL...
2026-02-28 13:08:33 [INFO] httpxthrottlecache.filecache.transport: cache_dir=C:\Users\Nutzer\.edgar\_tcache
2026-02-28 13:08:33 [INFO] httpxthrottlecache.filecache.transport: No cache policy for data.sec.gov:///submissions/CIK0000320193.json, not retrieving from cache
2026-02-28 13:08:33 [INFO] httpxthrottlecache.ratelimiter: Making HTTP Request <Request('GET', 'https://data.sec.gov/submissions/CIK0000320193.json')>
2026-02-28 13:08:34 [INFO] httpx: HTTP Request: GET https://data.sec.gov/submissions/CIK0000320193.json "HTTP/1.1 200 OK"
2026-02-28 13:08:34 [INFO] httpxthrottlecache.filecache.transport: No cache policy for data.sec.gov:///submissions/CIK0000320193-submissions-001.json, not retriev

Company: Apple Inc.
Filing Date: 2025-10-31
CIK: 320193
Full Text Length: 261,019 characters
Sections Extracted: 4


## 3. Inspect the Markdown Output

The generated Markdown has this structure:
```
# Apple Inc. — 10-K Annual Report
**Ticker:** AAPL
...
---
## Business
(content)
## Risk Factors
(content)
## MD&A
(content)
```

This structure enables **semantic chunking** by headers for RAG systems.

In [9]:
# Show all extracted sections and their sizes
print("Extracted Sections:")
print("-" * 50)
for section_name, content in filing.sections.items():
    print(f"  {section_name:45s} | {len(content):>8,} chars")
print("-" * 50)
print(f"  {'TOTAL':45s} | {sum(len(c) for c in filing.sections.values()):>8,} chars")

Extracted Sections:
--------------------------------------------------
  Business                                      |   16,054 chars
  Risk Factors                                  |   68,163 chars
  MD&A                                          |   18,018 chars
  Directors and Corporate Governance            |      401 chars
--------------------------------------------------
  TOTAL                                         |  102,636 chars


In [10]:
# Preview the generated Markdown (first 800 chars)
markdown_output = filing.to_markdown()
print("=== MARKDOWN OUTPUT (Preview) ===")
print(markdown_output[:800])
print("...")
print(f"\nTotal Markdown length: {len(markdown_output):,} chars")

=== MARKDOWN OUTPUT (Preview) ===
# Apple Inc. — 10-K Annual Report

**Ticker:** AAPL  
**CIK:** 320193  
**Filing Date:** 2025-10-31  
**Fiscal Year End:** 2025-09-27  
**Accession Number:** 0000320193-25-000079

---

## Business

Item 1.    Business

Company Background

The Company designs, manufactures and markets smartphones, personal computers, tablets, wearables and accessories, and sells a variety of related services. The Company’s fiscal year is the 52- or 53-week period that ends on the last Saturday of September.

Products

iPhone

iPhone® is the Company’s line of smartphones based on its iOS operating system. The iPhone line includes iPhone 17 Pro, iPhone Air™, iPhone 17, iPhone 16 and iPhone 16e.

Mac

Mac® is the Company’s line of personal computers based on its macOS® operating system. The Mac line includes l
...

Total Markdown length: 102,906 chars


## 4. Load a Previously Processed Filing

Once downloaded, filings are persisted as `.md` + `.meta.json` in `data/processed/`.  
They can be loaded without re-downloading from EDGAR.

In [11]:
from src.common import load_processed_filing

# Load from disk (no network call)
cached = load_processed_filing("AAPL")
print(f"Loaded: {cached.metadata.company_name} ({cached.metadata.filing_date})")
print(f"Sections: {list(cached.sections.keys())}")

Loaded: Apple Inc. (2025-10-31)
Sections: ['Business', 'Risk Factors', 'MD&A', 'Directors and Corporate Governance']


## 5. Batch Download All Configured Companies

The `download_all_filings()` function processes all companies defined in `configs/base.yaml`.  
Currently configured: **AAPL, MSFT, AMZN**.

In [12]:
from src.common import download_all_filings

# Download all 3 companies
results = download_all_filings()

print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
for f in results:
    print(f"  {f.metadata.ticker:6s} | {f.metadata.company_name:30s} | {len(f.sections)} sections | {len(f.full_text):>8,} chars")

2026-02-28 13:08:45 [INFO] src.common.ingestion: ==================================================
2026-02-28 13:08:45 [INFO] src.common.ingestion: Processing Apple Inc. (AAPL)
2026-02-28 13:08:45 [INFO] src.common.ingestion: ==================================================
2026-02-28 13:08:45 [INFO] edgar.core: Identity of the Edgar REST client set to [daniel.kojok@studmail.w-hs.de]
2026-02-28 13:08:45 [INFO] src.common.ingestion: EDGAR identity set to: daniel.kojok@studmail.w-hs.de
2026-02-28 13:08:45 [INFO] src.common.ingestion: Fetching 10-K filing for AAPL...
2026-02-28 13:08:45 [INFO] httpxthrottlecache.filecache.transport: cache_dir=C:\Users\Nutzer\.edgar\_tcache
2026-02-28 13:08:45 [INFO] httpxthrottlecache.filecache.transport: No cache policy for data.sec.gov:///submissions/CIK0000320193.json, not retrieving from cache
2026-02-28 13:08:45 [INFO] httpxthrottlecache.ratelimiter: Making HTTP Request <Request('GET', 'https://data.sec.gov/submissions/CIK0000320193.json')>
2026-0


SUMMARY
  AAPL   | Apple Inc.                     | 4 sections |  261,019 chars
  MSFT   | MICROSOFT CORP                 | 4 sections |  349,397 chars
  AMZN   | AMAZON COM INC                 | 4 sections |  330,396 chars
